# 04b The second encoder, fitted jointly with last year's volatility

`04-Second-encoder.ipynb` ran the two-step ridge and attention network of 02 and 03 on the chunk vectors of Yang et
al.'s FinBERT (CLS and token average). `02b` and `03c` replace the two-step fit (line first, text on its residual)
with a joint one, where last year's volatility and the text are fitted together. `03c` already redoes the ridge on
Yang's vectors that way; this notebook redoes the attention network, so the comparison of the two encoders is again
like for like.

The network is `joint.make_attention`, exactly as in 03c: the network of 03 plus one unpenalised weight on last
year's volatility, early stopping on the last training year, then a refit on the whole window, centred on the test
year so the level is the line's. The only difference from 03c is the vectors: Yang's, read from the shards 04 saved
on Google Drive. Both windows, five seeds, both kinds of vector.

**This needs a Colab GPU** (Runtime, Change runtime type, T4), about 30 minutes. Anywhere else it does nothing but
load the data.

On Colab, the first cell connects Google Drive, gets the repo and copies `data.pkl`. Locally it does nothing.

In [ ]:
import importlib.metadata, importlib.util, os, shutil, subprocess, sys
from pathlib import Path

NOTEBOOK_DIR = "Alex"
IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB:
    # data.pkl was written with pandas 3 and Colab ships pandas 2, which cannot read it. Install pandas 3 before
    # anything imports pandas. If Colab has already loaded pandas, the runtime has to restart: Run all again.
    if int(importlib.metadata.version("pandas").split(".")[0]) < 3:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas==3.0.6"], check=True)
        if "pandas" in sys.modules:
            print("pandas 3 installed but the old one was already loaded: restarting the runtime. Run all again.")
            os.kill(os.getpid(), 9)
    from google.colab import drive
    drive.mount("/content/drive")
    repo = Path("/content/dst-sprint2")
    if repo.exists():
        subprocess.run(["git", "-C", str(repo), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "https://github.com/Oscar339/dst-sprint2.git", str(repo)], check=True)
    processed = repo / "data" / "processed"
    processed.mkdir(parents=True, exist_ok=True)
    if not (processed / "data.pkl").exists():
        shutil.copy(Path("/content/drive/MyDrive/sprint2-data/data.pkl"), processed / "data.pkl")
    os.chdir(repo / NOTEBOOK_DIR)

In [ ]:
import pickle, platform, time
sys.path.insert(0, ".")

import numpy as np
import pandas as pd
import torch

import scoring
import attention
import joint

DEVICE = attention.DEVICE
print(f"Python {platform.python_version()} | torch {torch.__version__} | device {DEVICE}"
      + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else " (nothing to do here: run on a Colab GPU)"))
with open("../data/processed/data.pkl", "rb") as f:   # Oscar's hand-over file (Oscar/make_handover.py)
    d = pickle.load(f)
df = scoring.frame(d)
del d

WINDOWS = ["strict", "paper"]
DRIVE = Path("/content/drive/MyDrive/sprint2-data")
OUT = DRIVE / "second-encoder"
YANG_DIR = OUT / "yang-item7"
RESULTS_FILE, RUNS_FILE = OUT / "alex-04b-joint-results.csv", OUT / "alex-04b-joint-runs.csv"
print(f"{len(df):,} reports")

## 1. Yang's chunk vectors

The shards from 04, put back in `data.pkl`'s row order, one array of chunk vectors per report, as 04 does.

In [ ]:
YANG_CHUNKS = None
shards = sorted(YANG_DIR.glob("[12]*_[0-9][0-9][0-9].npz")) if IN_COLAB and YANG_DIR.exists() else []
if shards:
    per_key = {}
    for p in shards:
        z = np.load(p)
        cls_all, mean_all, ns = z["cls"], z["mean"], z["n"]
        starts = np.concatenate([[0], np.cumsum(ns)[:-1]])
        for k, s0, n in zip(z["keys"], starts, ns):
            per_key[str(k)] = (cls_all[s0:s0 + n], mean_all[s0:s0 + n])
    assert set(per_key) == set(df["key"]), "every report in data.pkl must have Yang vectors"
    YANG_CHUNKS = {v: [per_key[k][i] for k in df["key"]] for i, v in [(0, "cls"), (1, "mean")]}
    del per_key
    print(f"{len(shards)} shards, {sum(len(c) for c in YANG_CHUNKS['cls']):,} Item 7 chunks")
else:
    print("No Yang shards found (they are on Google Drive, from 04).")

## 2. The joint network (Colab GPU only)

Results are appended to Drive after each vector type and seed, so a rerun skips what is done. At the end the files
are copied into `data/output/` in the Colab copy of the repo; download them from Drive into `data/output/` to commit.

In [ ]:
RUN_FULL = IN_COLAB and DEVICE == "cuda" and YANG_CHUNKS is not None
if RUN_FULL:
    done = pd.read_csv(RESULTS_FILE)["model"].unique().tolist() if RESULTS_FILE.exists() else []
    t_start = time.perf_counter()
    for seed in attention.HP["seeds"]:
        for v in ["cls", "mean"]:
            names = {ver: f"joint attention, Item 7, Yang {v}, {ver}, seed {seed}" for ver in ["no refit", "refit"]}
            if all(n in done for n in names.values()):
                continue
            print(f"Yang {v}, seed {seed}   ({(time.perf_counter() - t_start) / 60:.0f} min so far)")
            log = []
            preds = joint.make_attention(df, {"Item 7": YANG_CHUNKS[v]}, seed=seed, log=log)
            res = pd.concat([scoring.evaluate(df, preds[ver], name=n, windows=WINDOWS) for ver, n in names.items()],
                            ignore_index=True)
            res.to_csv(RESULTS_FILE, mode="a", header=not RESULTS_FILE.exists(), index=False)
            pd.DataFrame(log).assign(vectors=f"Yang {v}").to_csv(RUNS_FILE, mode="a", header=not RUNS_FILE.exists(), index=False)
    for f in [RESULTS_FILE, RUNS_FILE]:
        shutil.copy(f, Path("../data/output") / f.name)
    print(f"done in {(time.perf_counter() - t_start) / 60:.0f} min: "
          f"{pd.read_csv(RESULTS_FILE)['model'].nunique()} runs, copied to data/output/")
else:
    print("RUN_FULL is False: nothing trained (needs a Colab GPU and the Yang shards on Drive).")

## 3. Results

Read from the saved files, so this section runs anywhere once they are in `data/output/`. The change in MSE against
the straight line, averaged over the six test years and the seeds, for the joint network on each encoder's vectors,
next to the joint ridge on the same vectors (03c) and the two-step network of 04.

In [ ]:
out = Path("../data/output")
def avg(t):
    t = t.assign(model=t["model"].str.replace(r", seed \d+$", "", regex=True))
    return t.groupby(["model", "window"], sort=False)["change_pct"].mean().unstack().round(2)

rows = []
for f, keep in [("alex-04b-joint-results.csv", None),
                ("alex-03c-joint-attention-results.csv", None),
                ("alex-03c-yang-joint-ridge-results.csv", None),
                ("alex-02b-joint-ridge-results.csv", ("joint within-year ridge, Item 7",)),
                ("alex-04-results.csv", None)]:
    if (out / f).exists():
        t = pd.read_csv(out / f)
        rows.append(avg(t if keep is None else t[t["model"].str.startswith(keep)]))
display(pd.concat(rows) if rows else "no results yet")